# 02 — Training Analysis

Post-training analysis of a completed fine-tuning run:
- Loss curves (train / eval)
- Gradient norm evolution
- Learning rate schedule visualisation
- VRAM usage over time
- Checkpoint quality comparison

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import seaborn as sns

from src.utils.checkpoint import list_checkpoints, best_checkpoint

sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.dpi'] = 120

OUTPUT_DIR = Path('../outputs/lora_7b')  # change to your run

In [ ]:
# Load trainer state
# Look for trainer_state.json in latest checkpoint or output dir
state_paths = sorted(OUTPUT_DIR.rglob('trainer_state.json'))
assert state_paths, f'No trainer_state.json found under {OUTPUT_DIR}'

# Use the most recent one
state = json.loads(state_paths[-1].read_text())
log_history = state['log_history']

print(f'Total log entries: {len(log_history)}')
print(f'Epochs trained:    {state["epoch"]:.2f}')
print(f'Max steps:         {state["max_steps"]}')
print(f'Log history keys:  {set().union(*[d.keys() for d in log_history])}')

In [ ]:
# Parse loss curves
train_steps, train_loss = [], []
eval_steps,  eval_loss  = [], []
grad_steps,  grad_norm  = [], []
lr_steps,    lr_values  = [], []

for entry in log_history:
    step = entry.get('step', 0)
    if 'loss' in entry:
        train_steps.append(step)
        train_loss.append(entry['loss'])
    if 'eval_loss' in entry:
        eval_steps.append(step)
        eval_loss.append(entry['eval_loss'])
    if 'grad_norm' in entry:
        grad_steps.append(step)
        grad_norm.append(entry['grad_norm'])
    if 'learning_rate' in entry:
        lr_steps.append(step)
        lr_values.append(entry['learning_rate'])

print(f'Train loss points: {len(train_loss)}')
print(f'Eval  loss points: {len(eval_loss)}')

In [ ]:
def smooth(values, window=10):
    """Simple moving average."""
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode='valid')


fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# --- Loss curves ---
ax1 = fig.add_subplot(gs[0, :])
if train_loss:
    ax1.plot(train_steps, train_loss, alpha=0.3, color='steelblue', label='Train (raw)')
    smooth_steps = train_steps[len(train_steps)-len(smooth(train_loss)):]
    ax1.plot(smooth_steps, smooth(train_loss), color='steelblue', lw=2, label='Train (smoothed)')
if eval_loss:
    ax1.plot(eval_steps, eval_loss, 'o-', color='coral', lw=2, label='Eval', markersize=5)
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training & Evaluation Loss')
ax1.legend()

# --- Gradient norm ---
ax2 = fig.add_subplot(gs[1, 0])
if grad_norm:
    ax2.plot(grad_steps, grad_norm, alpha=0.4, color='purple')
    if len(grad_norm) > 10:
        ax2.plot(grad_steps[len(grad_steps)-len(smooth(grad_norm)):],
                 smooth(grad_norm), color='purple', lw=2)
ax2.set_xlabel('Step')
ax2.set_ylabel('Gradient L2 Norm')
ax2.set_title('Gradient Norm')

# --- Learning rate ---
ax3 = fig.add_subplot(gs[1, 1])
if lr_values:
    ax3.plot(lr_steps, lr_values, color='green', lw=2)
    ax3.fill_between(lr_steps, lr_values, alpha=0.2, color='green')
ax3.set_xlabel('Step')
ax3.set_ylabel('Learning Rate')
ax3.set_title('LR Schedule')

plt.suptitle('Training Run Analysis', fontsize=14, fontweight='bold')
plt.savefig('training_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Checkpoint overview
checkpoints = list_checkpoints(OUTPUT_DIR)
print(f'Found {len(checkpoints)} checkpoints:')
for ckpt in checkpoints:
    print(f'  step={ckpt.step:6d}  epoch={ckpt.epoch:.2f}  eval_loss={ckpt.eval_loss}')

best = best_checkpoint(OUTPUT_DIR)
if best:
    print(f'\nBest checkpoint: {best.path.name} (eval_loss={best.eval_loss})')

In [ ]:
# Perplexity from loss
import math

if train_loss and eval_loss:
    train_ppl = [math.exp(min(l, 10)) for l in train_loss]
    eval_ppl  = [math.exp(min(l, 10)) for l in eval_loss]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(train_steps, train_ppl, alpha=0.35, color='steelblue')
    ax.plot(
        train_steps[len(train_steps)-len(smooth(train_ppl)):],
        smooth(train_ppl), color='steelblue', lw=2, label='Train PPL'
    )
    ax.plot(eval_steps, eval_ppl, 'o-', color='coral', lw=2, label='Eval PPL', markersize=5)
    ax.set_xlabel('Step')
    ax.set_ylabel('Perplexity')
    ax.set_title('Perplexity (exp(loss))')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f'Final train PPL: {train_ppl[-1]:.2f}')
    print(f'Best  eval  PPL: {min(eval_ppl):.2f}')